In [ ]:
import os, zipfile, glob
from google.colab import drive, userdata

drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
BASE = '/content/drive/MyDrive/adp_project'
OUT = '/content/bdd_subset'

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

!pip -q install ultralytics wandb
import ultralytics, wandb
print('ultralytics', ultralytics.__version__, '| wandb', wandb.__version__, '| torch', torch.__version__)

if not os.path.isdir(f'{OUT}/labels'):
    zipfile.ZipFile(f'{BASE}/data/bdd100k_subset.zip').extractall(OUT)
for sp in ('train', 'val', 'test'):
    n_img = len(glob.glob(f'{OUT}/images/{sp}/*.jpg'))
    n_lab = len(glob.glob(f'{OUT}/labels/{sp}/*.txt'))
    print(sp, 'images:', n_img, 'labels:', n_lab)
    assert n_img == n_lab > 0, f'{sp}: image/label mismatch'

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')   # never printed
wandb.login()
print('Setup OK')

In [ ]:
import time, os, glob
from ultralytics import YOLO, settings

assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
settings.update({'wandb': True})
os.environ['WANDB_PROJECT'] = 'adp-bdd100k'

t0 = time.time()
model = YOLO('yolo11s.pt')
res = model.train(
    data=f'{OUT}/data.yaml', epochs=1, fraction=0.1, imgsz=640, batch=16,
    workers=2, device=0, seed=42, plots=False,
    project=f'{BASE}/runs', name='smoke', exist_ok=True,
)
print(f'\nSMOKE TEST TIME: {time.time() - t0:.0f} s for 1 epoch on 400 images (includes model download and validation)')
print({k: round(v, 4) for k, v in res.results_dict.items() if 'mAP' in k})
print('saved weights:', glob.glob(f'{BASE}/runs/**/smoke*/weights/*.pt', recursive=True))

In [ ]:
import os, time
import wandb
from ultralytics import YOLO, settings

assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
assert os.path.isdir(f'{OUT}/labels'), 'Run the setup cell first'
settings.update({'wandb': True})

RUN = f'{BASE}/runs/yolo11s_bdd'
LAST = f'{RUN}/weights/last.pt'
EPOCHS = 30

wandb.init(project='adp-bdd100k', name='yolo11s-bdd-30ep',
           id='yolo11s-bdd-30ep-v1', resume='allow')

done = 0
if os.path.exists(f'{RUN}/results.csv'):
    done = sum(1 for _ in open(f'{RUN}/results.csv')) - 1
print('epochs already finished:', done)

t0 = time.time()
if done >= EPOCHS:
    print('Training already finished - skipping')
elif os.path.exists(LAST):
    print('Resuming from', LAST)
    try:
        YOLO(LAST).train(resume=True)
    except Exception as e:
        print('Resume problem:', type(e).__name__, str(e)[:300])
else:
    YOLO('yolo11s.pt').train(
        data=f'{OUT}/data.yaml', epochs=EPOCHS, patience=10, imgsz=640, batch=16,
        workers=2, cache='ram', device=0, seed=42, plots=True,
        project=f'{BASE}/runs', name='yolo11s_bdd', exist_ok=True,
    )
print(f'\nTRAINING CELL TIME: {(time.time() - t0) / 60:.1f} min')
print('weights in Drive:', [f for f in os.listdir(f'{RUN}/weights')] if os.path.isdir(f'{RUN}/weights') else 'none')
wandb.finish()

In [ ]:
import os, csv, zipfile, collections, glob
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from ultralytics import YOLO

BASE, OUT = '/content/drive/MyDrive/adp_project', '/content/bdd_subset'
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
if not os.path.isdir(f'{OUT}/labels'):
    zipfile.ZipFile(f'{BASE}/data/bdd100k_subset.zip').extractall(OUT)
W = f'{BASE}/runs/yolo11s_bdd/weights/best.pt'
assert os.path.exists(W), 'best.pt not found in Drive'
NAMES = ['person','rider','car','truck','bus','train','motor','bike','traffic light','traffic sign']
model = YOLO(W)

m = model.val(data=f'{OUT}/data.yaml', split='test', imgsz=640, batch=16, plots=False, verbose=False,
              project='/content/cond_runs', name='overall', exist_ok=True)
print(f'OVERALL TEST: mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}  recall={m.box.mr:.3f}')

meta = [r for r in csv.DictReader(open(f'{BASE}/logs/metadata.csv')) if r['split'] == 'test']
groups = collections.defaultdict(list)
for r in meta:
    groups['time: ' + r['timeofday']].append(r['image'])
    groups['weather: ' + r['weather']].append(r['image'])

MIN_IMAGES, res = 50, []
for i, (g, stems) in enumerate(sorted(groups.items())):
    if len(stems) < MIN_IMAGES or g.endswith(('undefined', 'foggy')):
        print('skipped:', g, len(stems)); continue
    lst, yml = f'/content/grp_{i}.txt', f'/content/grp_{i}.yaml'
    open(lst, 'w').write('\n'.join(f'{OUT}/images/test/{s}.jpg' for s in stems))
    open(yml, 'w').write(f'path: {OUT}\ntrain: images/train\nval: {lst}\nnames:\n' +
                         ''.join(f'  {k}: {n}\n' for k, n in enumerate(NAMES)))
    n_boxes = sum(1 for s in stems for ln in open(f'{OUT}/labels/test/{s}.txt') if ln.strip())
    gm = model.val(data=yml, split='val', imgsz=640, batch=16, plots=False, verbose=False,
                   project='/content/cond_runs', name=f'g{i}', exist_ok=True)
    res.append([g, len(stems), n_boxes, round(gm.box.map50, 3), round(gm.box.map, 3), round(gm.box.mr, 3)])

assert res, 'No groups evaluated - send me the skipped lines'
cond = pd.DataFrame(res, columns=['group', 'n_images', 'n_boxes', 'mAP50', 'mAP50-95', 'recall'])
print('\n', cond.to_string(index=False))
cond.to_csv('/content/condition_breakdown.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(cond['group'], cond['mAP50'], color='#2b6cb0')
for i, (v, n) in enumerate(zip(cond['mAP50'], cond['n_images'])):
    ax.text(i, v + 0.005, f'{v:.2f}\n(n={n})', ha='center', fontsize=8)
ax.set_ylabel('mAP50 (test split)'); ax.set_title('Detection accuracy by condition - test split')
plt.xticks(rotation=35, ha='right'); plt.tight_layout()
plt.savefig('/content/condition_breakdown.png', dpi=150); plt.show()
for f in ('/content/condition_breakdown.png', '/content/condition_breakdown.csv'):
    files.download(f)